# SynthWorlds-SM v5.1 — three-seed R1 multilingual pilot

This notebook runs the Llama 3.2 1B feasibility pilot described in `docs/report/R1.tex`: five-language composition SFT, a same-seed retain-only reference, pre-unlearning competence measurement, and English-only GA/GD/NPO only if the predeclared gate passes.

The dataset must be the hash-validated private v5.1 package with `sft_training/corpus.jsonl` and its manifest. The training and evaluation labels are separate. Generation reads prompts only; the scoring stage opens gold labels. Translation review is AI-only and is not native-speaker certification.

Keep the same Kaggle session through all stages. The run controls start `False`; turn on one stage at a time after reviewing its printed preflight. A failed competence gate is a valid pilot result and intentionally blocks unlearning.

## 1. Kaggle setup

In Notebook Settings choose a GPU and enable Internet. Under **Add-ons → Secrets**, enable `HF_TOKEN` with a Hugging Face read token for the account approved for `meta-llama/Llama-3.2-1B-Instruct`. The code never prints or writes the token. Use the interactive Notebook Editor: API-launched Kaggle jobs may not receive notebook secrets.

In [ ]:
import os, sys, json, hashlib, shutil, stat, zipfile, subprocess, textwrap, csv
from pathlib import Path, PurePosixPath

# Set one stage True only when intentionally starting it; defaults are safe.
RUN_SFT = False
RUN_BASELINE_EVALUATION = False
RUN_UNLEARNING = False
RUN_POST_EVALUATION = False

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Use one T4 from a T4x2 allocation.
INPUT = Path("/kaggle/input")
EXTRACT_ROOT = Path("/kaggle/working/synthworlds_v5_1_project")

# Accept either the private project ZIP or one directly attached project directory.
zip_candidates = []
for archive_path in INPUT.rglob("*.zip"):
    try:
        with zipfile.ZipFile(archive_path) as archive:
            if any(name.endswith("/scripts/train_sft_lora.py") or name == "scripts/train_sft_lora.py"
                   for name in archive.namelist()):
                zip_candidates.append(archive_path)
    except zipfile.BadZipFile:
        continue
if len(zip_candidates) > 1:
    raise RuntimeError(f"Attach one v5.1 project ZIP; found {len(zip_candidates)} matching archives.")
if zip_candidates:
    if EXTRACT_ROOT.exists():
        shutil.rmtree(EXTRACT_ROOT)
    EXTRACT_ROOT.mkdir(parents=True)
    with zipfile.ZipFile(zip_candidates[0]) as archive:
        for member in archive.infolist():
            relative = PurePosixPath(member.filename)
            mode = member.external_attr >> 16
            if relative.is_absolute() or ".." in relative.parts or stat.S_ISLNK(mode):
                raise RuntimeError(f"Unsafe path in attached dataset ZIP: {member.filename}")
        archive.extractall(EXTRACT_ROOT)
    roots = list(EXTRACT_ROOT.rglob("scripts/train_sft_lora.py"))
else:
    roots = list(INPUT.rglob("scripts/train_sft_lora.py"))
if len(roots) != 1:
    raise RuntimeError(f"Expected exactly one v5.1 project; found {len(roots)}")
PROJECT_ROOT = roots[0].parent.parent
TRAIN_SCRIPT = PROJECT_ROOT / "scripts/train_sft_lora.py"
UNLEARN_SCRIPT = PROJECT_ROOT / "scripts/unlearn_lora.py"
EVAL_SCRIPT = PROJECT_ROOT / "scripts/evaluate_pilot_three_probe.py"
SNAPSHOT_SCRIPT = PROJECT_ROOT / "scripts/kaggle_result_snapshot.py"
CORPUS = PROJECT_ROOT / "sft_training/corpus.jsonl"
CORPUS_MANIFEST = PROJECT_ROOT / "sft_training/manifest.json"
RELEASE_VALIDATION = PROJECT_ROOT / "sft_training/release_validation.json"
EVAL_ROOT = PROJECT_ROOT / "training_release_draft/evaluation"
PROMPTS = EVAL_ROOT / "prompts.jsonl"
LABELS = EVAL_ROOT / "labels.jsonl"  # Do not open before the scoring cell.

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as source:
        for block in iter(lambda: source.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def run_json(command):
    result = subprocess.run(command, check=True, text=True, capture_output=True)
    try:
        return json.loads(result.stdout)
    except json.JSONDecodeError as exc:
        raise RuntimeError(f"Expected JSON from command {command[1]}:\n{result.stdout[-3000:]}\n{result.stderr[-1000:]}") from exc

for required in (CORPUS, CORPUS_MANIFEST, RELEASE_VALIDATION, PROMPTS, LABELS,
                 TRAIN_SCRIPT, UNLEARN_SCRIPT, EVAL_SCRIPT, SNAPSHOT_SCRIPT):
    if not required.is_file():
        raise FileNotFoundError(required)
manifest = json.loads(CORPUS_MANIFEST.read_text(encoding="utf-8"))
release = json.loads(RELEASE_VALIDATION.read_text(encoding="utf-8"))
if manifest.get("do_not_train") is not False or release.get("status") != "PILOT_TRAINING_RELEASE_READY":
    raise RuntimeError("The attached SFT corpus has not passed the fail-closed release assembler.")
if manifest.get("corpus_sha256") != sha256_file(CORPUS):
    raise RuntimeError("SFT corpus SHA-256 does not match its manifest.")
if release.get("manifest_sha256") != sha256_file(CORPUS_MANIFEST):
    raise RuntimeError("Release validation does not bind the current SFT manifest.")
if manifest.get("evaluation", {}).get("prompts_sha256") != sha256_file(PROMPTS):
    raise RuntimeError("Evaluation prompts do not match the SFT release manifest.")
if not LABELS.is_file():
    raise FileNotFoundError(LABELS)
print("Project:", PROJECT_ROOT)
print("SFT rows:", manifest.get("total_rows"), "| corpus SHA-256:", manifest["corpus_sha256"])
print("Evaluation prompt SHA verified; labels are present and remain unopened until scoring.")
print("Translation review:", manifest.get("translation_review_mode"), "| human review:", manifest.get("human_translation_reviewed"))

In [ ]:
# Install the training stack without replacing Kaggle's CUDA-enabled PyTorch build.
requirements = PROJECT_ROOT / "requirements-sft.txt"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)
import torch, transformers, peft, accelerate, numpy
from importlib.metadata import version
if sys.version_info < (3, 11):
    raise RuntimeError(f"This locked Transformers 5.22.1 stack requires Python 3.11+; found {sys.version}")
expected = {"transformers": "5.22.1", "peft": "0.21.2", "accelerate": "1.15.0"}
actual = {name: version(name) for name in expected}
if actual != expected:
    raise RuntimeError(f"Training dependency lock mismatch: expected={expected}, actual={actual}")
GPU_AVAILABLE = torch.cuda.is_available()
if not GPU_AVAILABLE and any((RUN_SFT, RUN_BASELINE_EVALUATION, RUN_UNLEARNING, RUN_POST_EVALUATION)):
    raise RuntimeError("A Kaggle GPU is required for an enabled experiment stage. Enable GPU, then rerun.")
print("torch", torch.__version__, "| locked stack", actual)
print("CUDA", torch.version.cuda, "| GPU", torch.cuda.get_device_name(0) if GPU_AVAILABLE else "not enabled (dry setup only)")

# A tiny LoRA smoke test, followed by deterministic offline checks of the R1 analysis pipeline.
smoke_code = "import torch; from peft import LoraConfig,get_peft_model; m=get_peft_model(torch.nn.Sequential(torch.nn.Linear(4,4)),LoraConfig(r=2,lora_alpha=4,target_modules=['0'])); assert any('lora_A' in n for n,_ in m.named_parameters()); print('PEFT_LORA_PASS')"
smoke = subprocess.run([sys.executable, "-c", smoke_code], check=True, text=True, capture_output=True)
print(smoke.stdout.strip())
subprocess.run([sys.executable, str(PROJECT_ROOT / "scripts/test_r1_pilot_pipeline.py")], check=True)
print("Fixture checks passed. Their synthetic answers are not experiment results.")

In [ ]:
# Resolve HF access when run interactively. API-launched Kaggle jobs may not receive
# notebook secrets; safe setup remains usable with all RUN_* flags False.
HF_TOKEN = None
MODEL_REVISION = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    print("Kaggle secret is unavailable in this session:", type(exc).__name__)
if HF_TOKEN:
    from huggingface_hub import HfApi
    os.environ["HF_TOKEN"] = HF_TOKEN
    model_info = HfApi().model_info("meta-llama/Llama-3.2-1B-Instruct", token=HF_TOKEN)
    MODEL_REVISION = model_info.sha
    if not MODEL_REVISION or len(MODEL_REVISION) != 40:
        raise RuntimeError("Hugging Face did not return a 40-character immutable model revision.")
    print("Gated model access verified; model revision pinned:", MODEL_REVISION)
elif any((RUN_SFT, RUN_BASELINE_EVALUATION, RUN_UNLEARNING, RUN_POST_EVALUATION)):
    raise RuntimeError("A model stage is enabled but this session has no HF_TOKEN secret. Use the Notebook Editor with HF_TOKEN enabled.")
else:
    print("HF_TOKEN unavailable; no model download or training will run in this dry setup.")


## 2. Preflight and run controls

The design uses three seeds (42, 43, 44), full SFT on forget + retain + disjoint `train_only` rows, and a retain-only reference on retain + the same `train_only` rows. Unlearning is English first-hop only and is permitted only after all three full-SFT seeds pass the bilingual competence floor.

**Run order:** in the first setup code cell, set one `RUN_*` switch to `True`; rerun the settings cell and only that stage's cell. Keep the other switches `False`. `Run All` with defaults performs setup without training. HF secrets are required only for an enabled model stage. After each stage, download the `*_latest.zip` checkpoint from `/kaggle/working` or save a Kaggle notebook version with outputs. To resume in a fresh Kaggle session, attach exactly one checkpoint ZIP from the most advanced completed stage as a notebook input; setup verifies its file hashes and restores it under `/kaggle/working`. An incomplete run directory is deliberately rejected; inspect or move it aside before retrying.


In [ ]:
SEEDS = (42, 43, 44)
METHODS = ("ga", "gd", "npo")
RESULTS_ROOT = Path("/kaggle/working/synthworlds_r1_results")
SFT_ROOT = RESULTS_ROOT / "sft"
RETAIN_ROOT = RESULTS_ROOT / "retain_only"
PRED_ROOT = RESULTS_ROOT / "predictions"
BASELINE_DIR = RESULTS_ROOT / "baseline_scored"
TARGETS_FILE = BASELINE_DIR / "precompetent_english_forget_targets.json"
UNLEARN_ROOT = RESULTS_ROOT / "unlearning"
POST_DIR = RESULTS_ROOT / "post_scored"
for directory in (RESULTS_ROOT, SFT_ROOT, RETAIN_ROOT, PRED_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

def sha256_tree(path):
    path = Path(path)
    if path.is_file():
        return sha256_file(path)
    h = hashlib.sha256()
    for child in sorted(item for item in path.rglob("*") if item.is_file()):
        h.update(child.relative_to(path).as_posix().encode("utf-8"))
        h.update(bytes.fromhex(sha256_file(child)))
    return h.hexdigest()

def refuse_partial_run(output_dir, run_manifest):
    output_dir, run_manifest = Path(output_dir), Path(run_manifest)
    if not run_manifest.is_file() and output_dir.exists() and any(output_dir.iterdir()):
        raise RuntimeError(
            f"Incomplete run artifacts exist without a completion manifest: {output_dir}. "
            "Move this directory aside after inspecting it; the notebook will not reuse partial checkpoints."
        )

sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
from kaggle_result_snapshot import restore_snapshot, write_snapshot

snapshot_archives = sorted(INPUT.rglob("*_latest.zip"))
if snapshot_archives:
    restored = restore_snapshot(snapshot_archives, Path("/kaggle/working"), RESULTS_ROOT)
    print("Restored hash-verified stage snapshot:", restored["stage"], "| files:", len(restored["files"]))
else:
    print("No prior stage snapshot attached; starting a fresh results workspace.")

def checkpoint_results(stage_name):
    checkpoint_root = Path("/kaggle/working/synthworlds_stage_checkpoints")
    final_path = checkpoint_root / f"{stage_name}_latest.zip"
    write_snapshot(RESULTS_ROOT, final_path, stage_name)
    print("Recoverable stage snapshot:", final_path, "| bytes:", final_path.stat().st_size)
    return final_path

# The RUN_* controls are defined in the first setup code cell and are not reset here.
if any((RUN_SFT, RUN_BASELINE_EVALUATION, RUN_UNLEARNING, RUN_POST_EVALUATION)) and not MODEL_REVISION:
    raise RuntimeError("A model stage is enabled but no immutable HF revision is available.")

preflight_command = [sys.executable, str(TRAIN_SCRIPT), "--corpus", str(CORPUS),
    "--manifest", str(CORPUS_MANIFEST), "--output-dir", str(SFT_ROOT / "preflight")]
if MODEL_REVISION:
    preflight_command += ["--model-revision", MODEL_REVISION]
preflight = run_json(preflight_command)
print(json.dumps(preflight, indent=2))
if preflight.get("status") != "READY_TO_TRAIN":
    raise RuntimeError("SFT preflight is blocked; inspect the errors above before proceeding.")

In [ ]:
# Stage A: three full-SFT and three retain-only adapters. Run only after reading the preflight.
def validate_sft_run(run_path, output_dir, seed, splits, expected_variant):
    run_path, output_dir = Path(run_path), Path(output_dir)
    if not run_path.is_file():
        raise RuntimeError(f"Training ended without a run manifest: {run_path}")
    run = json.loads(run_path.read_text(encoding="utf-8"))
    adapter_dir = Path(run.get("adapter_dir", ""))
    expected = {
        "status": "TRAINING_COMPLETE", "training_started": True,
        "model_id": "meta-llama/Llama-3.2-1B-Instruct", "model_revision": MODEL_REVISION,
        "training_script_sha256": sha256_file(TRAIN_SCRIPT),
        "seed": seed, "corpus_sha256": manifest["corpus_sha256"],
        "corpus_manifest_sha256": sha256_file(CORPUS_MANIFEST),
        "training_variant": expected_variant, "included_splits": list(splits),
    }
    mismatches = [key for key, value in expected.items() if run.get(key) != value]
    if (mismatches or not adapter_dir.is_dir() or adapter_dir != output_dir / "adapter"
            or run.get("adapter_sha256") != sha256_tree(adapter_dir)):
        raise RuntimeError(f"SFT manifest/adapter validation failed at {run_path}; mismatches={mismatches}")
    return run

def ensure_sft_run(output_dir, seed, splits, expected_variant):
    output_dir = Path(output_dir)
    run_path = output_dir / "run_manifest.json"
    if run_path.is_file():
        run = validate_sft_run(run_path, output_dir, seed, splits, expected_variant)
        print("Reusing hash-validated run:", run_path)
        return run
    refuse_partial_run(output_dir, run_path)
    command = [sys.executable, str(TRAIN_SCRIPT), "--corpus", str(CORPUS), "--manifest", str(CORPUS_MANIFEST),
               "--output-dir", str(output_dir), "--include-splits", *splits, "--seed", str(seed),
               "--model-revision", MODEL_REVISION, "--train"]
    subprocess.run(command, check=True)
    run = validate_sft_run(run_path, output_dir, seed, splits, expected_variant)
    checkpoint_results("stageA_sft")
    return run

FULL_RUNS = {}
RETAIN_RUNS = {}
if RUN_SFT:
    if not GPU_AVAILABLE or not MODEL_REVISION:
        raise RuntimeError("SFT requires an active GPU and an authenticated, pinned HF model revision.")
    for seed in SEEDS:
        FULL_RUNS[seed] = ensure_sft_run(SFT_ROOT / f"seed_{seed}", seed,
            ("forget", "retain", "train_only"), "composition_sft")
        RETAIN_RUNS[seed] = ensure_sft_run(RETAIN_ROOT / f"seed_{seed}", seed,
            ("retain", "train_only"), "composition_retain_only")
else:
    print("SFT not started. Set RUN_SFT=True only when intentionally using Kaggle GPU time.")


In [ ]:
# Stage B: generate with prompts only, then score and apply the pre-unlearning gate.
# This cell opens labels only in the separate score command, after all generations finish.
def generate_model(model_name, adapter_dir=None, seed=None):
    prediction_path = PRED_ROOT / f"{model_name}.jsonl"
    command = [sys.executable, str(EVAL_SCRIPT), "generate", "--prompts", str(PROMPTS),
               "--output", str(prediction_path), "--model-name", model_name,
               "--model-revision", MODEL_REVISION, "--batch-size", "4", "--max-new-tokens", "64"]
    if adapter_dir is not None:
        command += ["--adapter-dir", str(adapter_dir), "--seed", str(seed)]
    subprocess.run(command, check=True)
    return prediction_path

FULL_MODELS = [f"sft_seed_{seed}" for seed in SEEDS]
RETAIN_MODELS = [f"retain_seed_{seed}" for seed in SEEDS]
BASELINE_PREDICTIONS = [PRED_ROOT / "pretrained.jsonl"] + [PRED_ROOT / f"{name}.jsonl" for name in FULL_MODELS + RETAIN_MODELS]
if RUN_BASELINE_EVALUATION:
    if not GPU_AVAILABLE or not MODEL_REVISION:
        raise RuntimeError("Baseline generation requires an active GPU and a pinned HF model revision.")
    if not all((SFT_ROOT / f"seed_{seed}/run_manifest.json").is_file() and
               (RETAIN_ROOT / f"seed_{seed}/run_manifest.json").is_file() for seed in SEEDS):
        raise FileNotFoundError("Complete all six SFT runs before baseline evaluation.")
    pretrained_prediction = generate_model("pretrained")
    full_runs = {seed: validate_sft_run(SFT_ROOT / f"seed_{seed}/run_manifest.json",
        SFT_ROOT / f"seed_{seed}", seed, ("forget", "retain", "train_only"), "composition_sft") for seed in SEEDS}
    retain_runs = {seed: validate_sft_run(RETAIN_ROOT / f"seed_{seed}/run_manifest.json",
        RETAIN_ROOT / f"seed_{seed}", seed, ("retain", "train_only"), "composition_retain_only") for seed in SEEDS}
    for seed in SEEDS:
        generate_model(f"sft_seed_{seed}", full_runs[seed]["adapter_dir"], seed)
        generate_model(f"retain_seed_{seed}", retain_runs[seed]["adapter_dir"], seed)
    baseline_predictions = [pretrained_prediction] + [PRED_ROOT / f"{name}.jsonl" for name in FULL_MODELS + RETAIN_MODELS]
    score_command = [sys.executable, str(EVAL_SCRIPT), "score", "--prompts", str(PROMPTS),
                     "--labels", str(LABELS), "--predictions", *map(str, baseline_predictions),
                     "--output-dir", str(BASELINE_DIR)]
    subprocess.run(score_command, check=True)
    gate_command = [sys.executable, str(EVAL_SCRIPT), "baseline-gate", "--case-metrics",
                    str(BASELINE_DIR / "case_metrics.jsonl"), "--full-sft-models", *FULL_MODELS,
                    "--output", str(TARGETS_FILE), "--minimum-cases", "15", "--minimum-groups", "8",
                    "--expected-cases", str(manifest["evaluation"]["evaluation_cases"])]
    subprocess.run(gate_command, check=True)
    gate = json.loads(TARGETS_FILE.read_text(encoding="utf-8"))
    print("Pre-unlearning competence gate:", gate["baseline_gate_status"])
    print(json.dumps(gate.get("bilingual_counts_by_language", {}), indent=2))
    checkpoint_results("stageB_baseline")
else:
    print("Baseline evaluation not started. Set RUN_BASELINE_EVALUATION=True after all six SFT runs are complete.")

In [ ]:
# Stage C: English-only first-hop GA/GD/NPO for all three matched seeds.
# A failed competence gate is recorded as a no-go; it never launches unlearning.
def validate_unlearning_run(run_path, output_dir, method, seed, sft_run_path, baseline_name):
    run_path, output_dir = Path(run_path), Path(output_dir)
    if not run_path.is_file():
        raise RuntimeError(f"Unlearning ended without a run manifest: {run_path}")
    run = json.loads(run_path.read_text(encoding="utf-8"))
    sft_run = json.loads(Path(sft_run_path).read_text(encoding="utf-8"))
    gate = json.loads(TARGETS_FILE.read_text(encoding="utf-8"))
    expected_checkpoint = gate["full_sft_artifacts"][baseline_name]["checkpoint_sha256"]
    adapter_dir = Path(run.get("adapter_dir", ""))
    expected = {
        "status": "UNLEARNING_COMPLETE", "training_started": True,
        "method": method.upper(), "seed": seed, "model_id": "meta-llama/Llama-3.2-1B-Instruct",
        "model_revision": MODEL_REVISION, "baseline_gate_status": "PASS",
        "baseline_checkpoint_sha256": expected_checkpoint,
        "sft_run_manifest_sha256": sha256_file(sft_run_path),
        "source_corpus_sha256": manifest["corpus_sha256"],
        "corpus_manifest_sha256": sha256_file(CORPUS_MANIFEST),
        "target_ids_manifest_sha256": sha256_file(TARGETS_FILE),
        "holdout_rows_used": 0, "translation_review_mode": "AI_ONLY",
        "human_translation_reviewed": False,
    }
    mismatches = [key for key, value in expected.items() if run.get(key) != value]
    if (mismatches or not adapter_dir.is_dir() or adapter_dir != output_dir / "adapter"
            or run.get("adapter_sha256") != sha256_tree(adapter_dir)
            or run.get("training_script_sha256") != sha256_file(UNLEARN_SCRIPT)):
        raise RuntimeError(f"Unlearning manifest/adapter validation failed at {run_path}; mismatches={mismatches}")
    return run

if RUN_UNLEARNING:
    if not GPU_AVAILABLE or not MODEL_REVISION:
        raise RuntimeError("Unlearning requires an active GPU and a pinned HF model revision.")
    if not TARGETS_FILE.is_file():
        raise FileNotFoundError("Run baseline evaluation and gate first.")
    gate = json.loads(TARGETS_FILE.read_text(encoding="utf-8"))
    if gate.get("baseline_gate_status") != "PASS" or gate.get("do_not_unlearn") is not False:
        print("UNLEARNING BLOCKED BY DESIGN: the pre-unlearning bilingual competence gate failed.")
    else:
        for method in METHODS:
            for seed in SEEDS:
                baseline_name = f"sft_seed_{seed}"
                sft_run_path = SFT_ROOT / f"seed_{seed}/run_manifest.json"
                validate_sft_run(sft_run_path, SFT_ROOT / f"seed_{seed}", seed,
                    ("forget", "retain", "train_only"), "composition_sft")
                output_dir = UNLEARN_ROOT / method / f"seed_{seed}"
                run_path = output_dir / "run_manifest.json"
                if run_path.is_file():
                    validate_unlearning_run(run_path, output_dir, method, seed, sft_run_path, baseline_name)
                    print("Reusing hash-validated unlearning run:", run_path)
                    continue
                refuse_partial_run(output_dir, run_path)
                command = [sys.executable, str(UNLEARN_SCRIPT), "--corpus", str(CORPUS),
                    "--corpus-manifest", str(CORPUS_MANIFEST), "--sft-run-manifest", str(sft_run_path),
                    "--target-ids-manifest", str(TARGETS_FILE), "--baseline-model-name", baseline_name,
                    "--output-dir", str(output_dir), "--method", method, "--seed", str(seed)]
                check = run_json(command)
                if check.get("status") != "READY_TO_UNLEARN":
                    raise RuntimeError(f"{method}/{seed} preflight blocked: {check.get('errors')}")
                subprocess.run(command + ["--train"], check=True)
                validate_unlearning_run(run_path, output_dir, method, seed, sft_run_path, baseline_name)
                checkpoint_results("stageC_unlearning")
else:
    print("Unlearning not started. It remains gated on the passing three-seed competence manifest.")


In [ ]:
# Stage D: generate post-unlearning outputs, rescore the exact paired checkpoints, and summarize three seeds.
if RUN_POST_EVALUATION:
    if not GPU_AVAILABLE or not MODEL_REVISION:
        raise RuntimeError("Post-unlearning generation requires an active GPU and a pinned HF revision.")
    gate = json.loads(TARGETS_FILE.read_text(encoding="utf-8"))
    if gate.get("baseline_gate_status") != "PASS":
        print("Post-unlearning evaluation skipped because the competence gate failed.")
    else:
        pairs = []
        post_predictions = []
        for method in METHODS:
            for seed in SEEDS:
                baseline_name = f"sft_seed_{seed}"
                run_path = UNLEARN_ROOT / method / f"seed_{seed}/run_manifest.json"
                if not run_path.is_file():
                    raise FileNotFoundError(f"Missing completed {method}/{seed} run: {run_path}")
                validate_unlearning_run(run_path, UNLEARN_ROOT / method / f"seed_{seed}",
                    method, seed, SFT_ROOT / f"seed_{seed}/run_manifest.json", baseline_name)
                post_name = f"{method}_seed_{seed}"
                run = json.loads(run_path.read_text(encoding="utf-8"))
                post_predictions.append(generate_model(post_name, run["adapter_dir"], seed))
                pairs.append({"method": method, "reference": baseline_name, "post": post_name})
                checkpoint_results("stageD_post_unlearning")
        pairs_path = RESULTS_ROOT / "model_pairs.json"
        pairs_path.write_text(json.dumps(pairs, indent=2) + "\n", encoding="utf-8")
        all_predictions = BASELINE_PREDICTIONS + post_predictions
        subprocess.run([sys.executable, str(EVAL_SCRIPT), "score", "--prompts", str(PROMPTS),
            "--labels", str(LABELS), "--predictions", *map(str, all_predictions),
            "--output-dir", str(POST_DIR)], check=True)
        EFFECTS_DIR = RESULTS_ROOT / "effects"
        subprocess.run([sys.executable, str(EVAL_SCRIPT), "effects", "--case-metrics",
            str(POST_DIR / "case_metrics.jsonl"), "--target-manifest", str(TARGETS_FILE),
            "--model-pairs", str(pairs_path), "--output-dir", str(EFFECTS_DIR), "--bootstrap", "10000"], check=True)
        print("Across-seed summary:", EFFECTS_DIR / "multi_seed_summary.csv")
else:
    print("Post-unlearning evaluation not started.")

In [ ]:
# Export concise, hash-bound outputs for Kaggle Save Version / download.
# Gold labels are included only in the scored experiment archive, never in generation inputs.
EXPORT_DIR = RESULTS_ROOT / "source_snapshot"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
for source in (TRAIN_SCRIPT, UNLEARN_SCRIPT, EVAL_SCRIPT,
               PROJECT_ROOT / "training_release_draft/scripts/assemble_training_release.py",
               PROJECT_ROOT / "training_release_draft/scripts/adjudicate_training_translations_qwen14b.py",
               PROJECT_ROOT / "scripts/test_r1_pilot_pipeline.py", SNAPSHOT_SCRIPT):
    shutil.copy2(source, EXPORT_DIR / source.name)
shutil.copy2(CORPUS_MANIFEST, EXPORT_DIR / "sft_manifest.json")
shutil.copy2(PROJECT_ROOT / "sft_training/release_validation.json", EXPORT_DIR / "release_validation.json")
shutil.copy2(PROMPTS, EXPORT_DIR / "evaluation_prompts.jsonl")
experiment_manifest = {
    "status": "R1_V5_1_KAGGLE_PILOT_ARTIFACTS",
    "model_id": "meta-llama/Llama-3.2-1B-Instruct", "model_revision": MODEL_REVISION,
    "seeds": list(SEEDS), "corpus_sha256": manifest["corpus_sha256"],
    "evaluation_prompts_sha256": sha256_file(PROMPTS),
    "translation_review_mode": "AI_ONLY", "human_translation_reviewed": False,
    "baseline_gate_status": json.loads(TARGETS_FILE.read_text())["baseline_gate_status"] if TARGETS_FILE.is_file() else "NOT_RUN",
    "post_unlearning_evaluation": (POST_DIR / "effect_analysis_manifest.json").is_file(),
}
(EXPORT_DIR / "experiment_manifest.json").write_text(json.dumps(experiment_manifest, indent=2) + "\n", encoding="utf-8")
RESULTS_ZIP = Path("/kaggle/working/synthworlds_r1_experiment_results.zip")
with zipfile.ZipFile(RESULTS_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RESULTS_ROOT.rglob("*")):
        if not path.is_file() or "checkpoint-" in path.parts:
            continue
        archive.write(path, path.relative_to(RESULTS_ROOT.parent))
print("Experiment status:", experiment_manifest["baseline_gate_status"])
print("Results archive:", RESULTS_ZIP, "| bytes:", RESULTS_ZIP.stat().st_size)

## Interpretation guardrails

- The primary scientific question is answered only if the competence gate passes and post-unlearning bilingual paired effects are available. A baseline gate failure means “not testable in this pilot,” not “no cross-language effect.”
- Retain-set results are training-overlap utility controls; they are not held-out generalization.
- The 1B run is a feasibility pilot, not the planned 8B main study or a power calculation.
- Translation review is AI-only. The Qwen reviewers share a model family; report this and do not call it human-validated.